In [107]:
import numpy as np
np.set_printoptions(suppress=True, precision=7)

In [108]:
def check_diag_dominant(matrix):
    sum_off_diagonal = np.sum(np.abs(matrix), axis=1) - np.abs(np.diag(matrix))
    diag = np.diag(matrix)
    compare = (sum_off_diagonal > diag).astype(int)

    print("Сума недіагональних:", sum_off_diagonal)
    print("Діагональні:", diag)
    print("Порушує діагональну перевагу (1 - так, 0 - ні):", compare)

In [109]:
A = np.array([
    [2.12, 0.42, 1.34, 0.88],
    [0.42, 3.95, 1.87, 0.43],
    [1.34, 1.87, 2.98, 0.46],
    [0.88, 0.43, 0.46, 4.44]
])

In [110]:
check_diag_dominant(A)

Сума недіагональних: [2.64 2.72 3.67 1.77]
Діагональні: [2.12 3.95 2.98 4.44]
Порушує діагональну перевагу (1 - так, 0 - ні): [1 0 1 0]


In [111]:
A = np.array([
    [2.12, 0.42, 1.34, 0.88],
    [0.42, 3.95, 1.87, 0.43],
    [1.34, 1.87, 2.98, 0.46],
    [0.88, 0.43, 0.46, 4.44]
])
b = np.array([11.172, 0.115, 0.009, 9.349])

A_1 = A.copy()
b_1 = b.copy()

A_1[0] = A_1[0] + (-1/2) * A_1[2]
b_1[0] = b_1[0] + (-1/2) * b_1[2]

print(A_1)
print(b_1)

[[ 1.45  -0.515 -0.15   0.65 ]
 [ 0.42   3.95   1.87   0.43 ]
 [ 1.34   1.87   2.98   0.46 ]
 [ 0.88   0.43   0.46   4.44 ]]
[11.1675  0.115   0.009   9.349 ]


In [112]:
check_diag_dominant(A_1)

Сума недіагональних: [1.315 2.72  3.67  1.77 ]
Діагональні: [1.45 3.95 2.98 4.44]
Порушує діагональну перевагу (1 - так, 0 - ні): [0 0 1 0]


In [113]:
A_1 = np.array([[ 1.45, -0.515, -0.15, 0.65],
 [0.42, 3.95, 1.87, 0.43],
 [1.34, 1.87, 2.98, 0.46],
 [0.88, 0.43, 0.46, 4.44]])


In [114]:

A_2 = A_1.copy()
b_2 = b_1.copy()

A_2[2] = A_2[2] + (-1/2) * A_2[0]
b_2[2] = b_2[2] + (-1/2) * b_1[0]

print(A_2)
print(b_2)

[[ 1.45   -0.515  -0.15    0.65  ]
 [ 0.42    3.95    1.87    0.43  ]
 [ 0.615   2.1275  3.055   0.135 ]
 [ 0.88    0.43    0.46    4.44  ]]
[11.1675   0.115   -5.57475  9.349  ]


In [115]:
check_diag_dominant(A_2)

Сума недіагональних: [1.315  2.72   2.8775 1.77  ]
Діагональні: [1.45  3.95  3.055 4.44 ]
Порушує діагональну перевагу (1 - так, 0 - ні): [0 0 0 0]


In [116]:
M = A_2

In [117]:
A_2, b_2

(array([[ 1.45  , -0.515 , -0.15  ,  0.65  ],
        [ 0.42  ,  3.95  ,  1.87  ,  0.43  ],
        [ 0.615 ,  2.1275,  3.055 ,  0.135 ],
        [ 0.88  ,  0.43  ,  0.46  ,  4.44  ]]),
 array([11.1675 ,  0.115  , -5.57475,  9.349  ]))

In [118]:
def find_C_and_d(A, b):
    C = np.zeros_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[0]):
            if i != j:
                C[i, j] = -A[i, j] / A[i, i]
    d = b / A.diagonal()
    return C, d

In [119]:
def simple_iteration(A, b, eps=1e-4, max_iter=100):
    C, d = find_C_and_d(A, b)
    q = np.max(np.sum(np.abs(C), axis=1))  # норма по рядках

    if q >= 1:
        print(f"Алгоритм не збіжний (q = {q:.4f} ≥ 1)")
        return None

    x_prev = np.zeros_like(b, dtype=float)

    for k in range(1, max_iter + 1):
        x_new = C @ x_prev + d

        delta = np.max(np.abs(x_new - x_prev))
        criterion = delta / (1 - q)

        # Вектор нев’язки
        residual = np.abs(b - A @ x_new)

        print(f"Ітерація {k}: x = {np.round(x_new, 6)}, критерій = {criterion:.6e}, r = {np.round(residual, 6)}")

        if criterion < eps:
            print(f"\nРозв’язок знайдено за {k} ітерацій:\n x = {np.round(x_new, 6)}")
            return x_new

        x_prev = x_new

    print("Досягнуто максимум ітерацій без збіжності.")
    return x_prev


In [120]:
res = simple_iteration(A_2, b_2)

Ітерація 1: x = [ 7.701724  0.029114 -1.824795  2.105631], критерій = 1.325564e+02, r = [1.627386 0.727778 5.08276  5.95063 ]
Ітерація 2: x = [ 6.579389 -0.155134 -3.488547  0.765399], критерій = 2.863527e+01, r = [0.526701 4.158895 1.263154 1.832207]
Ітерація 3: x = [ 6.942631  0.897751 -3.075076  1.178058], критерій = 1.812148e+01, r = [0.336028 1.103196 2.519115 0.96259 ]
Ітерація 4: x = [ 7.174374  0.618461 -3.899663  0.961258], критерій = 1.419220e+01, r = [0.126603 1.537871 0.480935 0.295471]
Ітерація 5: x = [ 7.087062  1.007796 -3.742238  1.027806], критерій = 6.700937e+00, r = [0.180865 0.28633  0.783596 0.162995]
Ітерація 6: x = [ 7.211797  0.935307 -3.998734  0.991095], критерій = 4.414623e+00, r = [0.051944 0.443045 0.082464 0.039392]
Ітерація 7: x = [ 7.175973  1.04747  -3.971741  0.999967], критерій = 1.930471e+00, r = [0.056046 0.039246 0.217793 0.029122]
Ітерація 8: x = [ 7.214626  1.037534 -4.043032  0.993408], критерій = 1.227005e+00, r = [0.011547 0.1199   0.001748 0.

In [121]:
7.22006
1.08331
-4.07652
0.992054

0.992054